# 📚 EduScan – VS Code / Local Version
**Run every cell top-to-bottom.  
Cell 2 will open a file-picker window so you can select your PDFs / images.**

| Step | What happens |
|------|-------------|
| Cell 1 | Verify conda env & install packages |
| Cell 2 | Config — opens file picker, sets output dirs, reads `HF_TOKEN` |
| Cell 3 | `qwen_api` module (Qwen2.5-7B via HF Router) |
| Cell 4 | Connection smoke-test |
| Cell 5 | `image_utils` — OCR engine |
| Cell 6 | `text_utils` — correction & summarization |
| Cell 7 | `index_utils` — RAG / ChromaDB |
| Cell 8 | `chat_utils` — logging |
| Cell 9 | **OCR pipeline** |
| Cell 10 | **Summarization** |
| Cell 11 | **Build RAG index** |
| Cell 12 | **Q&A** |
| Cell 13 | List output files |
| Cell 14 | Ad-hoc question |
| Cell 15 | Export → zip |


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 1 – Environment setup                                          ║
# ║                                                                      ║
# ║  ▶ Run ONCE in Anaconda Prompt before opening this notebook:         ║
# ║                                                                      ║
# ║    conda create -n mind python=3.10 -y                               ║
# ║    conda activate mind                                               ║
# ║    conda install -n mind ipykernel -y                                ║
# ║    python -m ipykernel install --user --name mind --display-name     ║
# ║             "Python (mind)"                                          ║
# ║                                                                      ║
# ║  Then in VS Code: select kernel  "Python (mind)"                     ║
# ╚══════════════════════════════════════════════════════════════════════╝

import subprocess, sys, importlib

PACKAGES = [
    # core
    "openai",
    "requests",
    "python-dotenv",
    # OCR
    "python-doctr[torch]",
    "Pillow",
    "numpy",
    "opencv-python",          # full GUI build for local use (NOT headless)
    "autocrop",
    # NLP
    "nltk",
    # RAG / embeddings
    "chromadb",
    "sentence-transformers",
    "huggingface_hub",
    # docs
    "python-docx",
]

print("Installing / verifying packages in the 'mind' environment...\n")
for pkg in PACKAGES:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q",
         "--disable-pip-version-check", pkg],
        capture_output=True, text=True,
    )
    real_errors = [
        ln for ln in result.stderr.splitlines()
        if ln.strip()
        and not ln.startswith("WARNING:")
        and not ln.startswith("DEPRECATION:")
    ]
    status = "✅" if result.returncode == 0 else "❌"
    print(f"  [{status}] {pkg}")
    for err in real_errors:
        print(f"        {err}")

print("\n[✅] All packages ready.")
print(f"     Python : {sys.executable}")
print(f"     Version: {sys.version.split()[0]}")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 – Configuration                                              ║
# ║  • Opens a file-picker window to select your PDFs / images           ║
# ║  • Reads HF_TOKEN from  .env  file next to this notebook             ║
# ║    (create it once:  echo HF_TOKEN=hf_xxxx > .env )                 ║
# ╚══════════════════════════════════════════════════════════════════════╝

import os, sys
from pathlib import Path
from datetime import datetime

# ── 1. Locate project root (folder containing this notebook) ──────────────
PROJECT_ROOT = Path().resolve()
print(f"[📁] Project root : {PROJECT_ROOT}")

# ── 2. Load HF_TOKEN from .env (preferred) or environment ─────────────────
try:
    from dotenv import load_dotenv
    env_file = PROJECT_ROOT / ".env"
    load_dotenv(dotenv_path=env_file, override=False)
    print(f"[🔑] Loaded .env  : {env_file}")
except ImportError:
    pass   # python-dotenv not installed yet — will be installed in Cell 1

HF_TOKEN = os.environ.get("HF_TOKEN", "").strip()
if HF_TOKEN:
    print("[✅] HF_TOKEN found.")
else:
    # Last resort: ask interactively (token won't be saved to disk)
    HF_TOKEN = input("⚠️  HF_TOKEN not found. Paste your token here: ").strip()
    os.environ["HF_TOKEN"] = HF_TOKEN
    print("[✅] HF_TOKEN set for this session (not saved).")

# ── 3. Open file-picker window ────────────────────────────────────────────
import tkinter as tk
from tkinter import filedialog

print("\n[📂] Opening file picker — select your PDFs / images…")
_root = tk.Tk()
_root.withdraw()          # hide the empty Tk window
_root.attributes("-topmost", True)   # make dialog appear on top

FILE_PATHS = list(filedialog.askopenfilenames(
    parent=_root,
    title="Select study files (PDF / JPG / PNG)",
    filetypes=[
        ("Study materials", "*.pdf *.jpg *.jpeg *.png"),
        ("PDF files",        "*.pdf"),
        ("Image files",      "*.jpg *.jpeg *.png"),
        ("All files",        "*.*"),
    ],
))
_root.destroy()

if FILE_PATHS:
    print(f"[✅] {len(FILE_PATHS)} file(s) selected:")
    for p in FILE_PATHS:
        print(f"     • {p}")
else:
    print("[⚠️]  No files selected. You can set FILE_PATHS manually below.")
    # FILE_PATHS = [r"C:\Users\you\notes.pdf"]   # ← manual fallback

# ── 4. Working directories (local, next to the notebook) ──────────────────
OUTPUT_DIR   = str(PROJECT_ROOT / "output")
RAG_DIR      = str(PROJECT_ROOT / "rag_index")
CHAT_LOG_DIR = str(PROJECT_ROOT / "chat_logs")
MODEL_DIR    = str(PROJECT_ROOT / "models" / "all-MiniLM-L6-v2")
TEMP_DIR     = str(PROJECT_ROOT / "temp")

for d in [OUTPUT_DIR, RAG_DIR, CHAT_LOG_DIR, MODEL_DIR, TEMP_DIR]:
    os.makedirs(d, exist_ok=True)

# ── 5. Questions for the RAG Q&A session ──────────────────────────────────
QUESTIONS = [
    "What are the main topics covered in these notes?",
    "Summarize the key concepts in bullet points.",
    "What definitions are given in the material?",
]

print("\n[✅] Configuration complete.")
print(f"     OUTPUT_DIR   : {OUTPUT_DIR}")
print(f"     RAG_DIR      : {RAG_DIR}")
print(f"     CHAT_LOG_DIR : {CHAT_LOG_DIR}")
print(f"     MODEL_DIR    : {MODEL_DIR}")
print(f"     FILE_PATHS   : {len(FILE_PATHS)} file(s)")
print(f"     QUESTIONS    : {len(QUESTIONS)} question(s) queued")


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  qwen_api  –  Production Cell                                        ║
# ║  Model   : Qwen/Qwen2.5-7B-Instruct                                  ║
# ║  SDK     : openai  (official HF-recommended client)                  ║
# ║  Endpoint: https://router.huggingface.co/v1  (HF Inference Router)   ║
# ╚══════════════════════════════════════════════════════════════════════╝

import os, time
from typing import Optional
from openai import OpenAI, APIStatusError, APITimeoutError, APIConnectionError

_QW_MODEL_ID     = "Qwen/Qwen2.5-7B-Instruct"
_QW_BASE_URL     = "https://router.huggingface.co/v1"
_QW_MAX_TOKENS   = 1024
_QW_TEMPERATURE  = 0.7
_QW_TOP_P        = 0.9
_QW_TIMEOUT      = 90
_QW_MAX_RETRIES  = 4
_QW_BACKOFF_BASE = 2.0
_QW_SYSTEM_PROMPT = "You are Qwen, created by Alibaba Cloud. You are a helpful assistant."
_QW_CLIENT: Optional[OpenAI] = None


def _get_client() -> OpenAI:
    global _QW_CLIENT
    if _QW_CLIENT is None:
        token = os.environ.get("HF_TOKEN", "").strip()
        if not token:
            raise EnvironmentError(
                "\n[qwen_api] ❌  HF_TOKEN is not set!\n"
                "  Fix: create a .env file next to this notebook with:\n"
                "       HF_TOKEN=hf_your_token_here\n"
                "  Then re-run Cell 2."
            )
        _QW_CLIENT = OpenAI(base_url=_QW_BASE_URL, api_key=token)
    return _QW_CLIENT


def generate_qwen_response(
    prompt: str,
    max_tokens: int = _QW_MAX_TOKENS,
    system_prompt: str = _QW_SYSTEM_PROMPT,
) -> str:
    client = _get_client()
    print(f"[🚀] Calling HF Router → {_QW_MODEL_ID}")

    for attempt in range(1, _QW_MAX_RETRIES + 1):
        backoff = _QW_BACKOFF_BASE ** (attempt - 1)
        try:
            completion = client.chat.completions.create(
                model=_QW_MODEL_ID,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user",   "content": prompt},
                ],
                max_tokens=max_tokens,
                temperature=_QW_TEMPERATURE,
                top_p=_QW_TOP_P,
                timeout=_QW_TIMEOUT,
            )
            answer = completion.choices[0].message.content.strip()
            usage  = getattr(completion, "usage", None)
            if usage:
                print(f"[✅] Done — prompt={usage.prompt_tokens} | "
                      f"completion={usage.completion_tokens} | "
                      f"total={usage.total_tokens} tokens")
            else:
                print(f"[✅] Response received ({len(answer)} chars).")
            return answer

        except APIStatusError as exc:
            code = exc.status_code
            if code == 503:
                try:    wait = max(float(exc.body.get("estimated_time", backoff)), backoff)
                except: wait = backoff
                print(f"[⏳] 503 – Model loading, waiting {wait:.0f}s… ({attempt}/{_QW_MAX_RETRIES})")
            elif code == 429:
                wait = backoff
                print(f"[🚦] 429 – Rate limited, backing off {wait:.0f}s… ({attempt}/{_QW_MAX_RETRIES})")
            else:
                msg = f"HTTP {code}: {str(exc)[:300]}"
                print(f"[❌] {msg}"); return f"Error: {msg}"
            if attempt < _QW_MAX_RETRIES:
                time.sleep(wait); continue
            return f"Error: HTTP {code} after {_QW_MAX_RETRIES} attempts."

        except APITimeoutError:
            print(f"[⏱️]  Timeout after {_QW_TIMEOUT}s ({attempt}/{_QW_MAX_RETRIES})")
            if attempt < _QW_MAX_RETRIES:
                time.sleep(backoff); continue
            return f"Error: Timeout after {_QW_MAX_RETRIES} attempts."

        except APIConnectionError as exc:
            msg = f"Connection error: {exc}"
            print(f"[❌] {msg}")
            if attempt < _QW_MAX_RETRIES:
                time.sleep(backoff); continue
            return f"Error: {msg}"

    return f"Error: All {_QW_MAX_RETRIES} attempts failed."


def ask_qwen(query: str, context: str) -> str:
    prompt = (
        "[CONTEXT]\n"
        f"{context.strip()}\n\n"
        "[QUESTION]\n"
        f"{query.strip()}\n\n"
        "Answer clearly and concisely."
    )
    return generate_qwen_response(prompt)


def test_qwen_connection() -> bool:
    print("[🔬] Testing connection to HF Inference Router…")
    result = generate_qwen_response("Reply with the single word: OK", max_tokens=10)
    ok   = not result.startswith("Error:")
    icon = "✅" if ok else "❌"
    print(f"[{icon}] Test {'PASSED' if ok else 'FAILED'} → {result[:80]}")
    return ok


print("[✅] qwen_api module ready.")
print(f"     Model    : {_QW_MODEL_ID}")
print(f"     Endpoint : {_QW_BASE_URL}")
print(f"     SDK      : openai  (HF Inference Router)")
print(f"     Retries  : {_QW_MAX_RETRIES}  |  Timeout: {_QW_TIMEOUT}s")


In [ ]:
# ── Verify HF_TOKEN works before running the full pipeline ────────────────
test_qwen_connection()


In [ ]:
import os, numpy as np
from PIL import Image
from autocrop import Cropper
from doctr.io import DocumentFile
from doctr.models import ocr_predictor

print("[🔄] Loading doctr OCR model (first run downloads weights)…")
_OCR_PREDICTOR = ocr_predictor(pretrained=True)
print("[✅] OCR model loaded.")


def auto_crop_image(image_path: str):
    """Auto-crop an image; fall back to the original if cropping fails."""
    print(f"[✂️]  Auto-cropping {image_path}…")
    try:
        cropped = Cropper().crop(image_path)
        if cropped is not None:
            img = Image.fromarray(cropped) if isinstance(cropped, np.ndarray) else cropped
            print("     Crop succeeded.")
            return img
        print("     Crop yielded no result; using original.")
        return Image.open(image_path)
    except Exception as exc:
        print(f"     Crop error ({exc}); using original.")
        try:
            return Image.open(image_path)
        except Exception as exc2:
            print(f"     Cannot open image: {exc2}")
            return None


def run_ocr(path: str):
    """Run doctr OCR on an image or PDF."""
    print(f"[🔍] Running OCR on {path}…")
    try:
        if not os.path.exists(path):
            print(f"[❌] File not found: {path}"); return None
        doc = (
            DocumentFile.from_images(path)
            if path.lower().endswith(("jpg", "jpeg", "png"))
            else DocumentFile.from_pdf(path)
        )
        result = _OCR_PREDICTOR(doc)
        print("     OCR completed.")
        return result
    except Exception as exc:
        print(f"[❌] OCR error: {exc}"); return None


def extract_text(result) -> str:
    """Flatten doctr OCR result to plain text."""
    print("[📝] Extracting text from OCR result…")
    if result is None: return ""
    try:
        lines = []
        for page in result.pages:
            for block in page.blocks:
                for line in block.lines:
                    lines.append(" ".join(w.value for w in line.words))
        text    = "\n".join(lines)
        preview = text[:120].replace("\n", " ")
        print(f"     Extracted {len(text)} chars | preview: {preview}…")
        return text
    except Exception as exc:
        print(f"[❌] Text extraction error: {exc}"); return ""


print("[✅] image_utils module ready.")


In [ ]:
import re, nltk

nltk.download("punkt",     quiet=True)
nltk.download("punkt_tab", quiet=True)


def correct_text(text: str) -> str:
    """Fix spelling/grammar and enrich content. Returns Markdown."""
    print("[✏️]  Correcting text…")
    prompt = (
        "Correct the spelling and grammar of this educational content. "
        "Also, expand on ideas and add more detail if possible to make the text "
        "clearer and richer. Return the output in well-structured Markdown format "
        "with headings, bullet points, or numbered lists where appropriate:\n\n"
        f"{text}"
    )
    return generate_qwen_response(prompt)


def summarize_text(text: str) -> str:
    """Produce a comprehensive Markdown summary."""
    print("[📌] Summarizing text…")
    prompt = (
        "Please write a detailed and comprehensive summary of the following "
        "educational text. Include all important points, explain concepts where "
        "possible, and keep it longer and informative. Return the output in "
        "well-structured Markdown format with headings, bullet points, or numbered "
        "lists where appropriate:\n\n"
        f"{text}"
    )
    return generate_qwen_response(prompt)


def tag_by_subject(text: str, subject_name: str) -> str:
    return f"[{subject_name}]\n{text}"


def segment_text(text: str) -> dict:
    if not text:
        return {"headings": [], "bullets": [], "other_text": []}
    sentences = nltk.tokenize.sent_tokenize(text)
    headings  = [s for s in sentences if s.isupper()]
    bullets   = [s for s in sentences if re.match(r"^[-*•]", s.strip())]
    other     = [s for s in sentences if s not in headings and s not in bullets]
    return {"headings": headings, "bullets": bullets, "other_text": other}


print("[✅] text_utils module ready.")


In [ ]:
import glob, chromadb
from sentence_transformers import SentenceTransformer
from huggingface_hub import snapshot_download

print("[🔄] Loading sentence-transformer embedding model…")
if not os.path.exists(os.path.join(MODEL_DIR, "config.json")):
    print("     Model not cached – downloading from HF Hub…")
    snapshot_download(
        repo_id="sentence-transformers/all-MiniLM-L6-v2",
        local_dir=MODEL_DIR,
    )

_EMBEDDING_MODEL = SentenceTransformer(MODEL_DIR)
print("[✅] Embedding model loaded.")


def build_or_load_index(folder: str = OUTPUT_DIR, persist_dir: str = RAG_DIR):
    print("[🗂️]  Building / loading RAG index…")
    docs = []
    for filepath in glob.glob(os.path.join(folder, "*.md")):
        with open(filepath, "r", encoding="utf-8") as fh:
            docs.append({"id": filepath, "text": fh.read()})

    client     = chromadb.PersistentClient(path=persist_dir)
    collection = client.get_or_create_collection("edu_docs")

    if docs:
        embeddings = [_EMBEDDING_MODEL.encode(d["text"]).tolist() for d in docs]
        collection.upsert(
            documents =[d["text"]           for d in docs],
            metadatas =[{"source": d["id"]} for d in docs],
            ids       =[d["id"]             for d in docs],
            embeddings=embeddings,
        )
        print(f"     Upserted {len(docs)} document(s) into '{persist_dir}'.")
    else:
        print(f"     No .md files found in '{folder}'. Index unchanged ({collection.count()} docs).")

    print(f"[✅] Index ready — {collection.count()} document(s) total.")
    return collection


def retrieve_passages(query: str, collection, top_k: int = 7) -> list:
    print(f"[🔎] Retrieving top {top_k} passages for: '{query}'")
    try:
        q_emb   = _EMBEDDING_MODEL.encode(query).tolist()
        results = collection.query(query_embeddings=[q_emb], n_results=top_k)
        flat    = []
        for group in results["documents"]:
            flat.extend(group if isinstance(group, list) else [group])
        print(f"     Retrieved {len(flat)} passage(s).")
        return flat
    except Exception as exc:
        print(f"[❌] Retrieval error: {exc}"); return []


print("[✅] index_utils module ready.")


In [ ]:
def save_chat_log(
    query: str,
    answer: str,
    log_directory: str = CHAT_LOG_DIR,
    log_filename:  str = "chat_history.md",
) -> None:
    os.makedirs(log_directory, exist_ok=True)
    filepath = os.path.join(log_directory, log_filename)
    try:
        with open(filepath, "a", encoding="utf-8") as fh:
            fh.write(f"### User:\n{query}\n\n### Assistant:\n{answer}\n\n---\n")
    except (PermissionError, IOError) as exc:
        print(f"[❌] Could not write chat log: {exc}")


def save_processed_outputs(summary: str, output_dir: str = OUTPUT_DIR) -> None:
    os.makedirs(output_dir, exist_ok=True)
    with open(os.path.join(output_dir, "summary.md"), "w", encoding="utf-8") as fh:
        fh.write(f"# Summary\n\n{summary or 'No summary generated.'}\n")
    print("[✅] summary.md saved.")


print("[✅] chat_utils module ready.")


In [ ]:
from datetime import datetime
from IPython.display import display, Markdown

if not FILE_PATHS:
    print("[⚠️]  FILE_PATHS is empty. Re-run Cell 2 to pick files.")
else:
    all_corrected_texts = {}

    for path in FILE_PATHS:
        print(f"\n{'='*60}")
        print(f"📄 Processing: {path}")
        print(f"{'='*60}")

        active_path = path

        # ── Auto-crop for images ───────────────────────────────────────
        if path.lower().endswith(("jpg", "jpeg", "png")):
            cropped = auto_crop_image(path)
            if cropped:
                if cropped.mode == "RGBA":
                    cropped = cropped.convert("RGB")
                # ✅ local temp path (not /kaggle/working)
                active_path = os.path.join(TEMP_DIR, "cropped_temp.jpg")
                cropped.save(active_path)
                print(f"     Saved cropped image → {active_path}")
            else:
                print("     Crop failed — using original path.")

        # ── OCR ───────────────────────────────────────────────────────
        ocr_res  = run_ocr(active_path)
        raw_text = extract_text(ocr_res)

        if not raw_text:
            print(f"[⚠️]  No text extracted from {path}. Skipping.")
            continue

        # ── Correction ────────────────────────────────────────────────
        corrected = correct_text(raw_text)
        if corrected.startswith("Error:"):
            print(f"[⚠️]  Correction failed: {corrected}")
            corrected = raw_text

        all_corrected_texts[path] = corrected

        # ── Save artefacts ────────────────────────────────────────────
        stem = os.path.basename(path).replace(" ", "_")
        with open(os.path.join(OUTPUT_DIR, f"{stem}_raw.md"),       "w", encoding="utf-8") as fh:
            fh.write(raw_text)
        with open(os.path.join(OUTPUT_DIR, f"{stem}_corrected.md"), "w", encoding="utf-8") as fh:
            fh.write(corrected)

        print(f"\n[📋] Corrected text preview (first 600 chars):")
        display(Markdown(corrected[:600] + "\n\n*…(truncated for preview)*"))

    print(f"\n[✅] OCR pipeline done. {len(all_corrected_texts)} file(s) processed.")


In [ ]:
summaries = {}

if not FILE_PATHS:
    print("[⚠️]  No files processed. Run Cell 9 first.")
else:
    for path, corrected in all_corrected_texts.items():
        print(f"\n[📌] Summarizing: {os.path.basename(path)}")
        summary = summarize_text(corrected)

        if summary.startswith("Error:"):
            print(f"[⚠️]  Summarization failed: {summary}")
            summary = "No summary generated."

        summaries[path] = summary
        stem = os.path.basename(path).replace(" ", "_")
        with open(os.path.join(OUTPUT_DIR, f"{stem}_summary.md"), "w", encoding="utf-8") as fh:
            fh.write(summary)

        print(f"\n[📋] Summary preview:")
        display(Markdown(summary[:800] + "\n\n*…(truncated for preview)*"))

    print(f"\n[✅] Summarization done. {len(summaries)} summary(ies) saved.")


In [ ]:
collection = build_or_load_index(folder=OUTPUT_DIR, persist_dir=RAG_DIR)
print(f"[✅] RAG index contains {collection.count()} document(s).")


In [ ]:
from datetime import datetime

if collection.count() == 0:
    print("[⚠️]  The RAG index is empty. Run Cell 11 first.")
elif not QUESTIONS:
    print("[⚠️]  QUESTIONS list is empty. Add questions in Cell 2.")
else:
    timestamp    = datetime.now().strftime("%Y%m%d_%H%M%S")
    log_filename = f"chat_{timestamp}.md"
    chat_history = []

    for q in QUESTIONS:
        print(f"\n{'─'*60}")
        print(f"❓ Question: {q}")
        print(f"{'─'*60}")

        passages = retrieve_passages(q, collection, top_k=7)

        if not passages:
            answer = "No relevant passages found in the index."
            print(f"[⚠️]  {answer}")
        else:
            context = "\n".join(passages)
            answer  = ask_qwen(q, context)

        print(f"\n💬 Answer:")
        display(Markdown(answer))

        save_chat_log(q, answer, log_filename=log_filename)
        chat_history.append({"question": q, "answer": answer})

    session_path = os.path.join(OUTPUT_DIR, "chat_session.md")
    with open(session_path, "w", encoding="utf-8") as fh:
        for entry in chat_history:
            fh.write(f"### User:\n{entry['question']}\n\n### Assistant:\n{entry['answer']}\n\n---\n")

    print(f"\n[✅] Q&A complete. {len(chat_history)} answer(s) saved to {session_path}")


In [ ]:
import glob as _glob

print(f"📂 {OUTPUT_DIR}")
for f in sorted(_glob.glob(os.path.join(OUTPUT_DIR, "*"))):
    size = os.path.getsize(f)
    print(f"   {os.path.basename(f):50s}  {size:>8,} bytes")

print(f"\n📂 {CHAT_LOG_DIR}")
for f in sorted(_glob.glob(os.path.join(CHAT_LOG_DIR, "*"))):
    size = os.path.getsize(f)
    print(f"   {os.path.basename(f):50s}  {size:>8,} bytes")


In [ ]:
# ── Edit the question and run this cell at any time ───────────────────────
CUSTOM_QUESTION = "What are the most important formulas in these notes?"
# ─────────────────────────────────────────────────────────────────────────

if collection.count() == 0:
    print("[⚠️]  Index is empty — run Cells 9–11 first.")
else:
    passages = retrieve_passages(CUSTOM_QUESTION, collection, top_k=7)
    context  = "\n".join(passages) if passages else "No relevant passages found."
    answer   = ask_qwen(CUSTOM_QUESTION, context)

    print(f"\n❓ {CUSTOM_QUESTION}\n")
    display(Markdown(answer))
    save_chat_log(CUSTOM_QUESTION, answer)


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════╗
# ║  📦  EXPORT  –  Zip all output files                                 ║
# ╚══════════════════════════════════════════════════════════════════════╝

import os, glob, zipfile, platform, subprocess
from datetime import datetime

EXPORT_PATH = os.path.join(
    PROJECT_ROOT,
    f"EduScan_Export_{datetime.now().strftime('%Y%m%d_%H%M%S')}.zip"
)

EXPORT_DIRS = {
    "output"    : OUTPUT_DIR,
    "chat_logs" : CHAT_LOG_DIR,
}

collected = []

with zipfile.ZipFile(EXPORT_PATH, "w", zipfile.ZIP_DEFLATED) as zf:
    for folder_name, folder_path in EXPORT_DIRS.items():
        for fpath in sorted(glob.glob(os.path.join(folder_path, "**", "*"), recursive=True)):
            if os.path.isfile(fpath):
                arcname = os.path.join(folder_name, os.path.relpath(fpath, folder_path))
                zf.write(fpath, arcname)
                collected.append((arcname, os.path.getsize(fpath)))

# ── Print manifest ─────────────────────────────────────────────────────
print(f"📦 {os.path.basename(EXPORT_PATH)}")
print(f"{'─'*60}")
for arcname, size in collected:
    print(f"  ✅  {arcname:<50}  {size:>8,} bytes")

zip_size = os.path.getsize(EXPORT_PATH)
print(f"{'─'*60}")
print(f"  📁  Total files : {len(collected)}")
print(f"  💾  Zip size    : {zip_size:,} bytes  ({zip_size/1024:.1f} KB)")
print(f"  📍  Saved to    : {EXPORT_PATH}")

# ── Open containing folder in OS file explorer ─────────────────────────
system = platform.system()
try:
    if system == "Windows":
        subprocess.run(["explorer", "/select,", EXPORT_PATH])
    elif system == "Darwin":     # macOS
        subprocess.run(["open", "-R", EXPORT_PATH])
    else:                        # Linux
        subprocess.run(["xdg-open", os.path.dirname(EXPORT_PATH)])
    print(f"\n[📂] File explorer opened at export location.")
except Exception as exc:
    print(f"\n[⚠️]  Could not open file explorer: {exc}")
    print(f"      Navigate manually to: {EXPORT_PATH}")
